In [44]:
import pandas as pd
import numpy as np

from tqdm import tqdm

In [45]:
TOP_ANIME_PATH = "top_animes.csv"
REPS_PATH = "spectral_13_clusters_5_reps.csv"
NUM_SAMPLES = 10
AMOUNT_OF_REPS_PER_CLUSTER = 5
DIFFERENCE = 10-1

In [46]:
top_animes = pd.read_csv(TOP_ANIME_PATH,keep_default_na=False,index_col="anime_id")
top_animes["number"] = np.arange(top_animes.shape[0])
reps = pd.read_csv(REPS_PATH,index_col = "anime_id")
train_values = np.load("training_vals.npy")

In [48]:
num_clusters = reps.cluster_id.max()
where = reps.cluster_id.values == np.arange(1,num_clusters+1)[:,np.newaxis]
low_val = -DIFFERENCE/num_clusters
high_val = DIFFERENCE+low_val
indices = top_animes.loc[reps.index,"number"].values
ratings= np.where(where,high_val,low_val)
print(ratings.shape)

(13, 65)


In [49]:
class MaximumStructure:
    # INITIAL = 200
    def __init__(self,keys):
        keep=keys>0
        self.inds = np.arange(keys.shape[0])
        self.inds= self.inds[keep]
        self.keys = keys[keep]
        self.shape = self.keys.shape[0]
        self.split = self.shape
        self.heapify()

    def swap(self,i,j):
        self.keys[i],self.keys[j] = self.keys[j],self.keys[i]
        self.inds[j],self.inds[i]=self.inds[i],self.inds[j]

    def sift_down(self,i):
        val_cur = self.keys[i]
        if 2*i+1<self.split:
            left_cur = self.keys[2*i+1]
        else:
            left_cur=-np.inf
        right_cur= self.keys[2*i+2] if 2*i+2<self.split else -np.inf
        if val_cur<right_cur and left_cur<=right_cur:
            selected=2*i+2
        elif val_cur<left_cur:
            selected=2*i+1
        else:
            return
        self.swap(i,selected)
        self.sift_down(selected)

    def heapify(self):
        # i = self.keys.argpartition(MaximumStructure.INITIAL)
        # self.keys=self.keys[i]
        # self.inds=self.inds[i]
        # i=np.argsort(self.keys[-MaximumStructure.INITIAL:])
        # self.keys[- MaximumStructure.INITIAL:]=self.keys[i]
        # self.inds[- MaximumStructure.INITIAL:]=self.inds[i]
        # self.split = self.shape - MaximumStructure.INITIAL
        for j in range((self.split-1)//2,-1,-1):
            self.sift_down(j)


    def extract_max(self):
        self.split-=1
        self.swap(0,self.split)
        self.sift_down(0)
        return self.inds[self.split],self.keys[self.split]

    def extracted(self):
        return self.inds[self.split:],self.keys[self.split:]

    def not_extracted(self):
        return self.split!=0

    def __iter__(self):
        for i in range(self.shape-1,self.split-1,-1):
            yield self.inds[i],self.keys[i]
        while self.split>-self.shape:
            yield self.extract_max()

In [50]:
similarities = np.inner(ratings,train_values[:,indices])
print(similarities.shape)

(13, 104303)


In [68]:
anime_to_predict = top_animes.drop(reps.index)
results = pd.DataFrame(columns = ("cluster_id",)+tuple(top_animes.columns))
with tqdm(total=num_clusters*anime_to_predict.shape[0]) as pbar:
    for i in range(num_clusters):
        similarity = similarities[i]
        similarity = MaximumStructure(similarity)

        #predicted ratings
        ratings  = np.repeat(-np.inf,top_animes.shape[0])
        for _,row in anime_to_predict.iterrows():
            num = row["number"]

            #find rating from the most similar user
            for ind,key in similarity:
                rat = train_values[ind,num]
                if rat!=0:
                    break
            else:
                continue
            #keep it
            ratings[num] = rat
            pbar.update(1)
        indices = np.argpartition(ratings,top_animes.shape[0]-NUM_SAMPLES)
        indices = indices[-NUM_SAMPLES:]

        #sort them by ratings in descending order
        ratings = ratings[indices]
        sorted_indices = indices[np.argsort(ratings)]
        anime_to_reccomend = top_animes.iloc[sorted_indices].copy()
        anime_to_reccomend["cluster_id"] = i+1
        results = pd.concat((results, anime_to_reccomend))

  7%|▋         | 1755/25155 [00:00<00:07, 3322.47it/s]C:\Users\Heart\AppData\Local\Temp\ipykernel_1472\1945460282.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat((results, anime_to_reccomend))


Index([32032, 49357, 48661, 12175, 18153, 122, 8129, 658, 39575, 31758], dtype='int64', name='anime_id')


 15%|█▍        | 3754/25155 [00:01<00:06, 3194.43it/s]

Index([38594, 9367, 9736, 19489, 2154, 7785, 22135, 440, 66, 35262], dtype='int64', name='anime_id')


 22%|██▏       | 5413/25155 [00:01<00:05, 3777.69it/s]

...

Index([40787, 37987, 3782, 37786, 5205, 14807, 2593, 35120, 67, 13357], dtype='int64', name='anime_id')


 91%|█████████▏| 22991/25155 [00:08<00:00, 3715.47it/s]

Index([7785, 38080, 31051, 57, 245, 30484, 1639, 48583, 22789, 4565], dtype='int64', name='anime_id')


100%|██████████| 25155/25155 [00:09<00:00, 2753.22it/s]

Index([339, 5114, 1210, 10161, 23755, 30484, 39741, 39535, 31758, 6213], dtype='int64', name='anime_id')


In [70]:
results.to_csv("cluster_other_representatives.csv",index_label="anime_id")